# Dual Next-Event Prediction Notebook

This notebook trains and compares dual-head models that predict:
- next activity (`concept:name`)
- next lifecycle transition (`lifecycle:transition`)

It runs two lifecycle data modes:
1. `start_complete` (only start/complete events)
2. `full_lifecycle` (all lifecycle transitions)

For each mode, it compares:
- `baseline`
- `balanced` (class-balanced sample weighting)

The final ranking is based on a balanced score.

In [10]:
from pathlib import Path
import sys
import json
import pandas as pd

# Make notebook imports robust to current working directory
cwd = Path.cwd().resolve()
if (cwd / "next_activity_prediction_lifecycle_dual").exists():
    repo_root = cwd
elif (cwd.parent / "next_activity_prediction_lifecycle_dual").exists():
    repo_root = cwd.parent
else:
    repo_root = cwd

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from next_activity_prediction_lifecycle_dual.config import DualPredictionConfig
from next_activity_prediction_lifecycle_dual.trainer import run_full_experiment

print("Notebook cwd:", cwd)
print("Repo root used for imports:", repo_root)

Notebook cwd: D:\Repos\process-simulation-engine-1\next_activity_prediction_lifecycle_dual
Repo root used for imports: D:\Repos\process-simulation-engine-1\next_activity_prediction_lifecycle_dual


In [ ]:
# Set your event log path here
# Examples:
# EVENT_LOG_PATH = "Dataset/BPIC12.xes"
# EVENT_LOG_PATH = "Dataset/your_log.csv"
# Set your event log path here
EVENT_LOG_PATH = fr"..\Dataset\BPI Challenge 2017.xes"

MODEL_ROOT = "next_activity_prediction_lifecycle_dual/models"

# Resolve relative path from detected repo root (set in previous cell)
log_path_obj = Path(EVENT_LOG_PATH)
if not log_path_obj.is_absolute():
    log_path_obj = repo_root / log_path_obj

if not log_path_obj.exists():
    raise FileNotFoundError(f"Event log not found: {log_path_obj}")

EVENT_LOG_PATH = str(log_path_obj)

config = DualPredictionConfig(
    sequence_length=50,
    embedding_dim=96,
    lstm_units=192,
    lstm_layers=2,
    dropout_rate=0.25,
    batch_size=64,
    learning_rate=0.001,
    epochs=1,
    validation_split=0.1,
    early_stopping_patience=8,
    model_root=MODEL_ROOT,
)

print("Using log:", EVENT_LOG_PATH)
print("Output root:", config.model_root)

Using log: D:\Repos\process-simulation-engine-1\next_activity_prediction_lifecycle_dual\..\Dataset\BPI Challenge 2017.xes
Output root: next_activity_prediction_lifecycle_dual\models


In [12]:
# Train all 4 combinations:
# - start_complete x baseline
# - start_complete x balanced
# - full_lifecycle x baseline
# - full_lifecycle x balanced
summary = run_full_experiment(log_path=EVENT_LOG_PATH, config=config)
summary["best_model"]

d:\Repos\process-simulation-engine-1\.venv\Lib\site-packages\pm4py\utils.py:987: UserWarning: In the current version, the import/export operation uses `rustxes` by default for importing/exporting files faster. Please uninstall `rustxes` to revert the behavior.
  warnings.warn("In the current version, the import/export operation uses `rustxes` by default for importing/exporting files faster. Please uninstall `rustxes` to revert the behavior.")


Epoch 1/40
2039/7545 ━━━━━━━━━━━━━━━━━━━━ 13:58 152ms/step - activity_output_loss: 1.0871 - activity_output_sparse_categorical_accuracy: 0.6463 - lifecycle_output_loss: 0.4166 - lifecycle_output_sparse_categorical_accuracy: 0.8198 - loss: 1.5037

KeyboardInterrupt: 

In [ ]:
summary_path = Path(MODEL_ROOT) / "comparison_summary.json"
with open(summary_path, "r", encoding="utf-8") as f:
    summary_from_file = json.load(f)

rows = []
for item in summary_from_file["all_results"]:
    m = item["metrics"]
    rows.append({
        "mode": item["mode"],
        "methodology": item["methodology"],
        "balanced_score": m["balanced_score"],
        "joint_accuracy": m["joint_accuracy"],
        "activity_macro_f1": m["activity_macro_f1"],
        "lifecycle_macro_f1": m["lifecycle_macro_f1"],
        "activity_accuracy": m["activity_accuracy"],
        "lifecycle_accuracy": m["lifecycle_accuracy"],
        "model_dir": item["model_dir"],
    })

results_df = pd.DataFrame(rows).sort_values("balanced_score", ascending=False)
results_df

In [ ]:
# Simple view: best model per lifecycle mode
best_per_mode = (
    results_df.sort_values("balanced_score", ascending=False)
    .groupby("mode", as_index=False)
    .first()
)
best_per_mode

## Notes

- Use `balanced_score` to choose the most well-balanced model.
- If you need faster iterations, reduce `epochs` in the config cell.
- You can rerun only the training cell after changing hyperparameters.
- Saved artifacts are in `next_activity_prediction_lifecycle_dual/models`.